
# Phase 3 — Model Development

This notebook evaluates **four machine learning algorithms**:

1. Logistic Regression (Baseline)
2. Decision Tree
3. Random Forest
4. Gradient Boosting

Workflow:

• Feature selection including **age, gender, department**  
• **Leakage-safe time-based train/test split**  
• **ColumnTransformer** preprocessing for categorical variables  
• **Pipeline integration**  
• Evaluation of all models  
• **Model comparison table**  
• **GridSearchCV hyperparameter tuning**  
• Saving the best model artifact


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import joblib


## Load Dataset

In [2]:
df = pd.read_csv("./data/model_table.csv")
df.head()


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approval_ratio,payment_delay_flag,visit_frequency,avg_los_per_patient,is_rejected,provider_rejection_rate,age_group,los_category,visit_intensity,dept_high_risk_rate
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.0,0,2,3.725000,1,0.148655,Senior,Short,0.030769,0.189950
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,1.0,0,4,32.025000,0,0.156915,Adult,Medium,-0.019417,0.202209
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,1.0,0,4,20.542500,0,0.149678,Adult,Long,0.444444,0.207923
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,1.0,0,7,28.165714,0,0.152480,Senior,Long,-0.112903,0.189950
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,1.0,0,5,22.988000,0,0.149678,Senior,Medium,5.000000,0.198439


## Feature Selection

In [3]:
target = "risk_score"

features = [
    "age",
    "gender",
    "department",
    "visit_type",
    "chronic_flag",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "days_since_registration"
    ]


## Time-based Train/Test Split

In [4]:
df = df.sort_values("visit_date")

split_index = int(len(df)*0.8)

train = df.iloc[:split_index]
test = df.iloc[split_index:]

X_train = train[features].fillna(0)
X_test = test[features].fillna(0)

y_train = train[target]
y_test = test[target]


## ColumnTransformer Preprocessing

In [5]:
categorical_features = ["gender", "department", "visit_type"]

numerical_features = [
    "age",
    "length_of_stay_hours",
    "visit_frequency",
    "avg_los_per_patient",
    "days_since_registration"

]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", "passthrough", numerical_features)
    ])


## Define Models

In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier()
}

results = []
best_model = None
best_score = 0


## Train and Evaluate Models

In [7]:
for name, model in models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict(X_test)

    acc = accuracy_score(y_test, preds)

    print("\nModel:", name)
    print("Accuracy:", acc)
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

    results.append((name, acc))

    if acc > best_score:
        best_score = acc
        best_model = pipeline



Model: Logistic Regression
Accuracy: 0.496
[[   0 1023    0]
 [   0 2480    0]
 [   0 1497    0]]
              precision    recall  f1-score   support

        High       0.00      0.00      0.00      1023
         Low       0.50      1.00      0.66      2480
      Medium       0.00      0.00      0.00      1497

    accuracy                           0.50      5000
   macro avg       0.17      0.33      0.22      5000
weighted avg       0.25      0.50      0.33      5000


Model: Decision Tree
Accuracy: 0.3712
[[ 224  434  365]
 [ 510 1126  844]
 [ 303  688  506]]


f:\AI ML\capstone\notebookenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\AI ML\capstone\notebookenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
f:\AI ML\capstone\notebookenv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


              precision    recall  f1-score   support

        High       0.22      0.22      0.22      1023
         Low       0.50      0.45      0.48      2480
      Medium       0.30      0.34      0.32      1497

    accuracy                           0.37      5000
   macro avg       0.34      0.34      0.34      5000
weighted avg       0.38      0.37      0.38      5000


Model: Random Forest
Accuracy: 0.4688
[[  37  791  195]
 [  65 2009  406]
 [  47 1152  298]]
              precision    recall  f1-score   support

        High       0.25      0.04      0.06      1023
         Low       0.51      0.81      0.62      2480
      Medium       0.33      0.20      0.25      1497

    accuracy                           0.47      5000
   macro avg       0.36      0.35      0.31      5000
weighted avg       0.40      0.47      0.40      5000


Model: Gradient Boosting
Accuracy: 0.4924
[[   5  918  100]
 [  13 2312  155]
 [  12 1340  145]]
              precision    recall  f1-score   

## Model Comparison Table

In [8]:
comparison = pd.DataFrame(results, columns=["Model","Accuracy"])
comparison.sort_values("Accuracy", ascending=False)


,Model,Accuracy
0,Logistic Regression,0.4960
3,Gradient Boosting,0.4924
2,Random Forest,0.4688
1,Decision Tree,0.3712


## Hyperparameter Tuning (Random Forest Example)

In [9]:
param_grid = {
    "model__n_estimators": [100, 200,500],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}



rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

grid = GridSearchCV(rf_pipeline,param_grid,cv=3,scoring="f1_macro",n_jobs=-1)

grid.fit(X_train,y_train)

print("Best Parameters:",grid.best_params_)

best_model = grid.best_estimator_


Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}


## Final Evaluation

In [10]:
preds = best_model.predict(X_test)

print("Best ModeL:",best_model)
print("Final Accuracy:", accuracy_score(y_test,preds))
print(confusion_matrix(y_test,preds))
print(classification_report(y_test,preds))


Best ModeL: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'department',
                                                   'visit_type']),
                                                 ('num', 'passthrough',
                                                  ['age',
                                                   'length_of_stay_hours',
                                                   'visit_frequency',
                                                   'avg_los_per_patient',
                                                   'days_since_registration'])])),
                ('model', RandomForestClassifier(random_state=42))])
Final Accuracy: 0.4688
[[  37  791  195]
 [  65 2009  406]
 [  47 1152  298]]
              precision    recall  f1-score   support

        High       0.25      0.0

## Save Model Artifact

In [11]:
joblib.dump(best_model,r".\data\risk_model.pkl",compress=3)
print("risk_model.pkl saved")


risk_model.pkl saved


In [12]:
import json

# Explicitly define the features and their types/values as required
feature_schema = {
    "age": {"dtype": "numeric"},
    "gender": {"dtype": "category", "values": ["F", "M"]},
    "department": {"dtype": "category", "values": ["Cardiology", "ER", "General", "ICU", "Neurology", "Orthopedics"]},
    "visit_type": {"dtype": "category", "values": ["ER", "ICU", "OPD"]},
    "chronic_flag": {"dtype": "numeric"},
    "length_of_stay_hours": {"dtype": "numeric"},
    "visit_frequency": {"dtype": "numeric"},
    "avg_los_per_patient": {"dtype": "numeric"},
    "days_since_registration": {"dtype": "numeric"}
}

schema = {
    "model_name": "hospital_prediction_model",
    "features": feature_schema
}

with open(r".\data\risk_features.json", "w") as f:
    json.dump(schema, f, indent=4)

print("risk_features.json created successfully")


risk_features.json created successfully
